# Binary Image Segmentation with VGG16 U-Net — Dust Storm Detection

End-to-end pipeline: data preprocessing → augmentation → model training → inference.

**Dataset:** ELAI Dust Storm Dataset from MODIS  
**Framework:** TensorFlow / Keras  
**Architecture:** VGG16 encoder + U-Net decoder with skip connections

---


## 0. Install Dependencies

In [ ]:
# Run this cell once to install required packages
# Comment out if already installed in your environment
# !pip install tensorflow[and-cuda]==2.17.1   # GPU (WSL2 / Linux)
# !pip install tensorflow==2.17.1             # CPU only (Windows)
!pip install opencv-python==4.10.0.84
!pip install scikit-learn==1.6.0
!pip install matplotlib==3.10.0
!pip install imgaug
!pip install tqdm


## 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import imgaug.augmenters as iaa
from tqdm import tqdm
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, Activation,
    Conv2DTranspose, Concatenate, Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG16

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Configuration

Edit `DATASET_ROOT` and `OUTPUT_DIR` to match your local paths before running.


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
DATASET_ROOT  = "/mnt/d/Data-Sets-Object-Segmentation/ELAI Dust Storm Dataset from MODIS"
OUTPUT_DIR    = "/mnt/d/temp"
MODEL_DIR     = "/mnt/d/temp/models/Dust-Storm"
MODEL_PATH    = os.path.join(MODEL_DIR, "VGG16-Dust-Storm.keras")

# ── Image dimensions ─────────────────────────────────────────────────────
HEIGHT, WIDTH = 128, 128   # Reduce if memory errors occur

# ── Training hyperparameters ─────────────────────────────────────────────
BATCH_SIZE    = 4           # Increase for GPUs with >12 GB VRAM
EPOCHS        = 200
LEARNING_RATE = 1e-4
RANDOM_SEED   = 42
THRESHOLD     = 0.5         # Inference binarisation threshold

# ── Derived paths ────────────────────────────────────────────────────────
IMAGES_PATH   = os.path.join(DATASET_ROOT, "images")
MASKS_PATH    = os.path.join(DATASET_ROOT, "annotations")
NPY_IMAGES    = os.path.join(OUTPUT_DIR, "Dust-Storm-Images.npy")
NPY_MASKS     = os.path.join(OUTPUT_DIR, "Dust-Storm-Masks.npy")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)
print("Config OK.")


## 3. Data Sanity Check

Load one image/mask pair to verify paths, colour channels, and mask pixel values.


In [ ]:
# Load a sample pair
sample_img  = cv2.cvtColor(
    cv2.imread(os.path.join(IMAGES_PATH, "13.jpg"), cv2.IMREAD_COLOR),
    cv2.COLOR_BGR2RGB
)
sample_mask = cv2.imread(os.path.join(MASKS_PATH, "13_GT.png"), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample_img);               axes[0].set_title("Original Image"); axes[0].axis("off")
axes[1].imshow(sample_mask, cmap="gray"); axes[1].set_title("Mask (raw 0/255)"); axes[1].axis("off")
plt.tight_layout()
plt.show()

# Verify mask pixel values before and after binarisation
m16 = cv2.resize(sample_mask, (16, 16))
print("Unique values BEFORE binarisation:", np.unique(m16))
m16[m16 > 0] = 1
print("Unique values AFTER  binarisation:", np.unique(m16))


## 4. Augmentation Preview

Preview the three augmentation types applied during preprocessing: horizontal flip, vertical flip, rotation.


In [ ]:
aug_hflip = iaa.Fliplr(p=1.0)
aug_vflip = iaa.Flipud(p=1.0)
aug_rot   = iaa.Affine(rotate=(-50, 20))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, title, img_data in zip(
    axes,
    ["Original", "H-Flip", "V-Flip", "Rotation"],
    [
        sample_img,
        aug_hflip.augment_image(sample_img),
        aug_vflip.augment_image(sample_img),
        aug_rot.augment_image(sample_img),
    ]
):
    ax.imshow(img_data)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Preprocessing & Augmentation — Build NumPy Arrays

For each image:
1. Load RGB image → resize → normalise to [0, 1]
2. Load grayscale mask → resize → binarise to {0, 1}
3. Apply 3 augmentations (H-flip, V-flip, rotation) → 4× dataset size
4. Save as `.npy` for fast training


In [ ]:
all_images, all_masks = [], []

image_files = [f for f in os.listdir(IMAGES_PATH)
               if os.path.isfile(os.path.join(IMAGES_PATH, f))]
print(f"Found {len(image_files)} raw images.")

for fname in tqdm(image_files, desc="Processing"):
    img_path  = os.path.join(IMAGES_PATH, fname)
    stem, _   = os.path.splitext(fname)
    mask_path = os.path.join(MASKS_PATH, f"{stem}_GT.png")

    # Image: BGR → RGB → resize → normalise
    img  = cv2.cvtColor(cv2.imread(img_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, (WIDTH, HEIGHT))
    img  = (img / 255.0).astype(np.float32)

    # Mask: grayscale → resize → binarise
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (WIDTH, HEIGHT))
    mask[mask > 0] = 1

    # Original + 3 augmentations
    aug_rot_instance = iaa.Affine(rotate=(-50, 20))

    for aug_img, aug_mask in [
        (img,                               mask),
        (aug_hflip.augment_image(img),      aug_hflip.augment_image(mask)),
        (aug_vflip.augment_image(img),      aug_vflip.augment_image(mask)),
        (aug_rot_instance.augment_image(img), aug_rot_instance.augment_image(mask)),
    ]:
        all_images.append(aug_img)
        all_masks.append(aug_mask)

images_np = np.array(all_images, dtype=np.float32)
masks_np  = np.array(all_masks,  dtype=int)

print(f"\nTotal samples after augmentation : {len(images_np)}")
print(f"Images shape : {images_np.shape}   dtype: {images_np.dtype}")
print(f"Masks  shape : {masks_np.shape}    dtype: {masks_np.dtype}")

np.save(NPY_IMAGES, images_np)
np.save(NPY_MASKS,  masks_np)
print(f"\nSaved NumPy arrays to {OUTPUT_DIR}")


## 6. Load NumPy Arrays & Train / Validation Split (80 / 20)


In [ ]:
images_np = np.load(NPY_IMAGES)
masks_np  = np.load(NPY_MASKS)
print(f"Loaded — Images: {images_np.shape}  Masks: {masks_np.shape}")

X_train, X_val, y_train, y_val = train_test_split(
    images_np, masks_np,
    test_size=0.2,
    random_state=RANDOM_SEED
)
print(f"Train : {X_train.shape}   Val : {X_val.shape}")


## 7. VGG16 U-Net Architecture

- **Encoder**: Pre-trained VGG16 (ImageNet weights, `include_top=False`)  
- **Skip connections**: feature maps from blocks 1–4 are concatenated into the decoder  
- **Decoder**: 4× `ConvTranspose2d` + conv blocks restoring full spatial resolution  
- **Output**: `sigmoid` activation → per-pixel probability of dust storm class


In [ ]:
def conv_block(inputs, num_filters):
    """Two consecutive Conv2D + BatchNorm + ReLU.""\"
    x = Conv2D(num_filters, 3, padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def decoder_block(inputs, skip_features, num_filters):
    """Upsample → concat skip → conv block.""\"
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(inputs)
    x = Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x


def build_vgg16_unet(input_shape):
    """Build VGG16-encoder U-Net for binary segmentation.""\"
    inputs = Input(input_shape)

    vgg16 = VGG16(include_top=False, weights="imagenet", input_tensor=inputs)

    # Encoder skip outputs
    s1 = vgg16.get_layer("block1_conv2").output   # full res
    s2 = vgg16.get_layer("block2_conv2").output   # /2
    s3 = vgg16.get_layer("block3_conv3").output   # /4
    s4 = vgg16.get_layer("block4_conv3").output   # /8

    # Bottleneck
    b1 = vgg16.get_layer("block5_conv3").output   # /16

    # Decoder
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    return Model(inputs, outputs, name="VGG16_U-Net")


model = build_vgg16_unet((HEIGHT, WIDTH, 3))
model.summary()


## 8. Compile & Train

**Loss:** Binary Cross-Entropy (matches sigmoid output + binary {0,1} targets)  
**Callbacks:**
- `ModelCheckpoint` — saves best model (lowest `val_loss`)
- `ReduceLROnPlateau` — halves LR if `val_loss` plateaus for 3 epochs
- `EarlyStopping` — stops training after 20 stagnant epochs


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=3, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor="val_loss", patience=20, verbose=1),
]

steps_per_epoch  = int(np.ceil(len(X_train) / BATCH_SIZE))
validation_steps = int(np.ceil(len(X_val)   / BATCH_SIZE))

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)


## 9. Learning Curves

In [ ]:
acc      = history.history["accuracy"]
val_acc  = history.history["val_accuracy"]
loss     = history.history["loss"]
val_loss = history.history["val_loss"]
ep_range = range(len(acc))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_range, acc,     "r", label="Train Accuracy")
axes[0].plot(ep_range, val_acc, "b", label="Val Accuracy")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].set_title("Training vs Validation Accuracy")
axes[0].legend(loc="lower right")

axes[1].plot(ep_range, loss,     "r", label="Train Loss")
axes[1].plot(ep_range, val_loss, "b", label="Val Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Binary Cross-Entropy")
axes[1].set_title("Training vs Validation Loss")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "learning_curves.png"), dpi=150)
plt.show()
print(f"Best model saved to: {MODEL_PATH}")


## 10. Inference on a Single Test Image

1. Load saved model
2. Preprocess test image identically to training (resize → normalise → expand dims)
3. Predict probability map
4. Threshold at 0.5 → binary mask
5. Visualise original image, probability heatmap, and binary mask side-by-side


In [ ]:
TEST_IMAGE_PATH  = "dust_storm_test_img.jpg"   # ← replace with your test image
OUTPUT_MASK_PATH = "dust_storm_test_mask.jpg"

# Load model
model_infer = tf.keras.models.load_model(MODEL_PATH)

# Load & preprocess test image
img_bgr = cv2.imread(TEST_IMAGE_PATH, cv2.IMREAD_COLOR)
if img_bgr is None:
    raise FileNotFoundError(f"Image not found: {TEST_IMAGE_PATH}")

img_rgb       = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized   = cv2.resize(img_rgb, (WIDTH, HEIGHT))
img_norm      = (img_resized / 255.0).astype(np.float32)
img_for_model = np.expand_dims(img_norm, axis=0)   # (1, H, W, 3)

# Predict
pred     = model_infer.predict(img_for_model)
prob_map = pred[0]   # (H, W, 1)
print(f"Probability range: [{prob_map.min():.3f}, {prob_map.max():.3f}]")

# Threshold → binary mask
binary_mask = prob_map.copy()
binary_mask[binary_mask <= THRESHOLD] = 0
binary_mask[binary_mask >  THRESHOLD] = 255
binary_mask = binary_mask.squeeze().astype(np.uint8)

# Resize mask back to original image size for display
orig_h, orig_w = img_rgb.shape[:2]
mask_disp = cv2.resize(binary_mask, (orig_w, orig_h), interpolation=cv2.INTER_AREA)
cv2.imwrite(OUTPUT_MASK_PATH, mask_disp)

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb);                        axes[0].set_title("Original Image");          axes[0].axis("off")
axes[1].imshow(prob_map.squeeze(), cmap="hot"); axes[1].set_title("Probability Map (heatmap)"); axes[1].axis("off")
axes[2].imshow(mask_disp, cmap="gray");         axes[2].set_title(f"Binary Mask (threshold={THRESHOLD})"); axes[2].axis("off")
plt.tight_layout()
plt.savefig("dust_storm_inference_result.png", dpi=150)
plt.show()
print(f"Mask saved to: {OUTPUT_MASK_PATH}")
